In [ ]:
import re
import pandas as pd
import parselmouth

def get_length_of_gs(tg: parselmouth.TextGrid) -> float:
    '''
    Returns length of a first glottal stop in a given textgrid
    '''
    num_intervals = parselmouth.praat.call(tg, "Get number of intervals", 1)
    for interval_num in range(1, num_intervals + 1):
        label = parselmouth.praat.call(tg, "Get label of interval", 1, interval_num).strip()
        if label == 'ʔ':
            start_time = parselmouth.praat.call(tg, "Get start point", 1, interval_num)
            end_time = parselmouth.praat.call(tg, "Get end point", 1, interval_num)
            break
    try:
        return end_time - start_time
    except UnboundLocalError:
        return None

def get_lenghts_of_gs_over_collection(filename: str, pattern: str):
    '''
    Returns dictionary of lengths of glottal stops with a given (regex) pattern in a given .collection
    '''
    res = dict()
    data = parselmouth.praat.call('Read from file...', filename)
    for obj in data:
        if type(obj) == parselmouth.TextGrid and re.search(pattern, obj.name) and obj.name.count('ʔ') == 1:
            length = get_length_of_gs(obj)
            if length:
                res[obj.name] = get_length_of_gs(obj)
    return res

def get_lengths_df(list_of_filenames: list[str], pattern):
    res = pd.DataFrame()
    for filename in list_of_filenames:
        df = pd.DataFrame.from_dict(get_lenghts_of_gs_over_collection(filename, pattern), orient='index')
        res = pd.concat((res, df))
    return res
    
list_of_filenames = ['backup10-1.Collection', 'backupДОПа10-1.Collection', 'доп2-бэкап10-1.Collection', 'доп3.Collection'] # файлы в которые я сохранял записи

intvocal = get_lengths_df(list_of_filenames, r'[aeiou]ʔ[aeiou]')
c_gs_v = get_lengths_df(list_of_filenames, r'[^$aeiouə]ʔ[aeiou]')

In [3]:
intvocal[0].mean()

np.float64(0.12118270553720938)

In [10]:
intvocal

,0
atxsanoʔan,0.100264
iʔiɬ_tɬinwilicen,0.183179
kzəmpleʔin,0.123353
kʼŋezeʔin,0.169932
qaʔaq,0.183368
tχaltχal_kotake_kəzzuʔin,0.051875
uʔin,0.121335
xiʔikəmlʲaχ,0.009063
tχiʔin,0.153285
atnoʔan,0.111025


In [5]:
c_gs_v[0].mean()

np.float64(0.10221665812816633)

In [11]:
c_gs_v

,0
isx_ənčoɬnen_qəllal_čajʔasx,0.112691
čɬxʔin,0.197056
əlxʔes,0.153917
ajwanxʔal_kkʼoɬknen,0.054163
kpaxalʔan,0.109325
tnumxʔal,0.052867
tnumʔin,0.094656
xanxʔal,0.087199
klačʔin,0.140943
qtoŋʔan,0.112392


In [8]:
from scipy import stats
stats.ttest_ind(intvocal[0], c_gs_v[0], equal_var=False)

TtestResult(statistic=np.float64(1.2093062901917593), pvalue=np.float64(0.2328999071704718), df=np.float64(44.71935595246229))